# LAB1 – Bài 7: Biến đổi các thuộc tính theo đề bài

**Họ và tên:** TRẦN TRUNG THÔNG  
**MSSV:** 6451071075  
**Môn học:** Khai thác dữ liệu  
**Nguồn bài tập:** Tài liệu LAB1, mục IV.2 – Yêu cầu lập trình (trang 15–27 theo trình đọc PDF).  
**Dữ liệu:** `titanic.csv` do giảng viên cung cấp (1.309 dòng, 14 thuộc tính).

> Mỗi bài là một notebook độc lập. Upload `titanic.csv` vào Colab, chạy từ trên xuống, bổ sung nhận xét theo kết quả thực tế và lưu notebook sau khi chạy.


In [ ]:
# Thông tin sinh viên (in bằng lệnh print theo yêu cầu của giảng viên)
HO_TEN = "TRẦN TRUNG THÔNG"
MSSV = "6451071075"
print("Họ và tên:", HO_TEN)
print("MSSV:", MSSV)
print("LAB1 - Bài 7: Biến đổi các thuộc tính theo đề bài")


## 1. Đọc dữ liệu và chuẩn bị thư viện

Tải `titanic.csv` vào bảng **Tệp** của Google Colab trước khi chạy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)

# Tải file titanic.csv của giảng viên lên Colab trước khi chạy.
# Có thể tải tại /content/titanic.csv hoặc /titanic.csv như phiên làm việc trước.
paths = ['/content/titanic.csv', '/titanic.csv', 'titanic.csv', 'titanic3.csv']
titanic_path = next((Path(p) for p in paths if Path(p).is_file()), None)
if titanic_path is None:
    raise FileNotFoundError('Không tìm thấy titanic.csv. Vào thanh Tệp của Colab và tải file lên trước.')

df = pd.read_csv(titanic_path)
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={'survival': 'survived'})
for col in ['pclass', 'survived', 'age', 'sibsp', 'parch', 'fare']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print('Đã đọc:', titanic_path)
print('Kích thước dữ liệu:', df.shape)
assert df.shape == (1309, 14), 'Hãy kiểm tra lại file Titanic do giảng viên cung cấp'


## Câu 7 — Biến đổi 5 thuộc tính

Cần: `fare` thành số nguyên; `title` tách từ `name`; `sex` và `embarked` thành số; `age` chia 8 nhóm; `fare` chia 6 nhóm.

**Lưu ý học thuật:** chuyển trực tiếp `fare` từ float sang int sẽ **mất phần thập phân**. Giữ lại `fare_original` để phục vụ câu 8. Việc mã hóa C/Q/S thành 0/1/2 chỉ để đáp ứng câu này, không nên dùng như thang đo cho mô hình khoảng cách; xem mã hóa one-hot ở phần cơ bản.

**Để số phần tử nhóm cân bằng hơn**: dùng `pd.qcut()` (thuộc **pandas**, không thuộc sklearn). Nếu nhiều giá trị trùng mốc phân vị, `duplicates='drop'` có thể tạo **ít hơn** 8 hoặc 6 nhóm; đây là tính chất dữ liệu, không phải lỗi code. Kết quả `qcut` là cách **thay thế**, không nhất thiết giống các mốc gợi ý trong tài liệu.

## 2. Thực hiện bài toán

In [ ]:
# Bước chuẩn bị riêng cho bài 7: tiền xử lý dữ liệu trước khi biến đổi thuộc tính.
df_rel = df.copy()
df_rel['relatives'] = df_rel['sibsp'].fillna(0) + df_rel['parch'].fillna(0)
df_rel['not_alone'] = (df_rel['relatives'] > 0).astype(int)

In [ ]:
df_clean = df_rel.copy()
# Tách chữ đầu tiên trong mã cabin: ví dụ C123 -> C; không có cabin -> U.
if 'cabin' in df_clean:
    df_clean['deck'] = df_clean['cabin'].astype('string').str.extract(r'^([A-Za-z])', expand=False).fillna('U')
else:
    df_clean['deck'] = 'U'

# Loại bỏ cột không cần thiết nếu chúng có trong bộ dữ liệu.
df_clean = df_clean.drop(columns=['ticket','boat','body','home.dest','cabin'], errors='ignore')

# Điền thiếu
if df_clean['age'].notna().any():
    df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
else:
    raise ValueError('Cột age không có giá trị hợp lệ để tính trung vị')
if df_clean['embarked'].notna().any():
    df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])
else:
    df_clean['embarked'] = 'Missing'
df_clean['fare'] = df_clean['fare'].fillna(df_clean['fare'].median())

print('Các cột còn lại:', df_clean.columns.tolist())
print('Số ô thiếu còn lại trên từng cột:')
display(df_clean.isna().sum().sort_values(ascending=False).head(12))
display(df_clean.head())

In [ ]:
# Giữ nguyên giá trị gốc
df_clean["fare_original"] = df_clean["fare"].copy()

In [ ]:
# 7a. Giá vé: chuyển float -> int (bỏ phần thập phân như ví dụ PDF)
df_clean['fare'] = df_clean['fare'].astype(int)

# 7b. Title: chữ giữa dấu phẩy và dấu chấm, vd "Allen, Miss. Elisabeth" -> "Miss"
df_clean['title_raw'] = df_clean['name'].astype('string').str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
common_titles = {'Mr', 'Miss', 'Mrs', 'Master'}
df_clean['title'] = df_clean['title_raw'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})
df_clean['title'] = df_clean['title'].where(df_clean['title'].isin(common_titles), 'Rare')

# 7c. Mã hóa sex và embarked
df_clean['sex_code'] = df_clean['sex'].map({'male': 0, 'female': 1})
df_clean['embarked_code'] = df_clean['embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# 7d. Chia tuổi thành 8 nhóm theo mốc gợi ý trong PDF
# Mốc đúng 11 tuổi được xếp vào Nhóm 1 để không bị bỏ sót.
# Các nhóm tuổi bằng mốc cố định KHÔNG bảo đảm số người mỗi nhóm bằng nhau.
df_clean['age'] = df_clean['age'].astype(int)
age_edges = [-np.inf, 11, 18, 22, 27, 33, 40, 66, np.inf]
df_clean['age_group'] = pd.cut(df_clean['age'], bins=age_edges, labels=list(range(1,9))).astype(int)

# 7e. Chia giá vé thành 6 nhóm theo mốc gợi ý
fare_edges = [-np.inf, 7.91, 14.454, 31, 99, 250, np.inf]
df_clean['fare_group'] = pd.cut(
    df_clean['fare_original'], bins=fare_edges, labels=list(range(1,7))
).astype(int)

print('Phân bố 8 nhóm tuổi:')
display(df_clean['age_group'].value_counts().sort_index())
print('Phân bố 6 nhóm giá vé:')
display(df_clean['fare_group'].value_counts().sort_index())
display(df_clean[['name','title','sex','sex_code','embarked','embarked_code',
                  'age','age_group','fare','fare_original','fare_group']].head(12))

In [ ]:
age_qcut = pd.qcut(df_clean['age'], q=8, duplicates='drop')
fare_qcut = pd.qcut(df_clean['fare_original'], q=6, duplicates='drop')
print('Nhóm tuổi theo phân vị (qcut):')
display(age_qcut.value_counts().sort_index())
print('Nhóm giá vé theo phân vị (qcut):')
display(fare_qcut.value_counts().sort_index())

## 3. Nhận xét kết quả

**Bạn viết nhận xét dựa trên số liệu và biểu đồ thực tế ở trên.** Không kết luận từ các nhóm có rất ít quan sát.

**Chụp màn hình:** Đảm bảo ảnh chụp thấy code, kết quả và **thanh taskbar Windows** ở dưới cùng. Nên chụp nhiều ảnh nếu kết quả dài.